<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# 4. RAG Pipeline with ChromaDB
**Industry:** Healthcare

A RAG system that answers clinical questions from a custom protocol document.

In [1]:
!pip install langchain langchain-openai chromadb sentence-transformers langchain-community

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-5.6.1-py3-none-any.whl.metadata (18 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pybase64-1.4.3-cp313-cp313-win_amd64.whl.metadata (9.1 kB)
  Using cached uvicorn-0.52.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached numpy-2.5.1-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.28.0-cp313-cp313-win_amd64.whl.metadata (5.7 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-no


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# Create a mock clinical protocol document
with open('protocol.txt', 'w') as f:
    f.write("""Clinical Protocol for Hypertension:\n1. First-line treatment for uncomplicated hypertension is an ACE inhibitor or ARB.\n2. If patient is over 55 or of African family origin, use Calcium Channel Blocker (CCB) first.\n3. Target blood pressure is <140/90 mmHg for most patients.""")

loader = TextLoader("protocol.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n"""
prompt = ChatPromptTemplate.from_template(template)
llm = AzureChatOpenAI(azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o"), api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"))

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the first-line treatment for a 60-year-old patient?"
print(rag_chain.invoke(question))

ModuleNotFoundError: No module named 'langchain_community'